Before removing the unnecessary geographical columns, I tried to recover missing values using the information already available in the original dataset. I checked the relationships between geographical variables, starting from more specific ones like `postal_code` and moving towards broader areas.

I only used matches that were unique in the dataset, even if supported by just one customer, and skipped ambiguous cases to avoid introducing incorrect values. This allowed me to recover missing information without randomly filling NaNs.

I also checked for duplicate rows and duplicated `customer_id` values to make sure the dataset was consistent before moving to the next preprocessing step.

In [18]:

import pandas as pd
from google.colab import files

# 1. Read original dataset
df = pd.read_csv("clientes.csv")
original = df.copy()

# 2. Geographical relationships
relationships = [
    ("postal_code", "postal_area"),
    ("postal_code", "municipality"),
    ("postal_code", "commercial_region"),
    ("postal_code", "country_or_region"),
    ("address", "postal_code"),
    ("address", "postal_area"),
    ("address", "municipality"),
    ("street", "postal_area"),
    ("street", "municipality"),
    ("postal_area", "municipality"),
    ("postal_area", "commercial_region"),
    ("postal_area", "country_or_region"),
    ("city", "municipality"),
    ("city", "commercial_region"),
    ("municipality", "commercial_region"),
    ("municipality", "country_or_region"),
    ("island", "country_or_region"),
    ("commercial_region", "country_or_region")
]

# 3. Build mappings from original data
lookups = {}

for source, target in relationships:
    known = original[[source, target]].dropna()

    lookups[(source, target)] = (
        known.groupby(source)[target]
        .agg(lambda x: set(x))
        .to_dict()
    )

# 4. Reconstruct missing values
for target in set(t for s, t in relationships):

    for idx in original.index[original[target].isna()]:

        candidates = set()
        ambiguous = False

        for source, t in relationships:
            if t != target:
                continue

            source_value = original.at[idx, source]

            if pd.isna(source_value):
                continue

            possible = lookups[(source, target)].get(
                source_value, set()
            )

            if len(possible) > 1:
                ambiguous = True
            elif len(possible) == 1:
                candidates.update(possible)

        # Fill only if the match is unique and unambiguous
        if not ambiguous and len(candidates) == 1:
            df.at[idx, target] = next(iter(candidates))

# 5. Compare missing values
summary = pd.DataFrame({
    "Before": original.isna().sum(),
    "After": df.isna().sum()
})

summary["Recovered"] = summary["Before"] - summary["After"]

print("Dataset shape:", df.shape)
print("\nReconstruction summary:")
display(summary)

print("Total recovered:", summary["Recovered"].sum())

# 6. Check duplicates
print("\n--- Duplicate Check ---")

# Completely duplicated rows
print("Duplicated rows:", df.duplicated().sum())

# Duplicated customer IDs
print("Duplicated customer IDs:",
      df["customer_id"].duplicated().sum())

# Display duplicated customers, if any
duplicates = df[
    df.duplicated(subset=["customer_id"], keep=False)
]

if not duplicates.empty:
    print("\nCustomers with duplicated IDs:")
    display(duplicates.sort_values("customer_id"))
else:
    print("No duplicated customer IDs found.")

# 7. Save and download reconstructed dataset
df.to_csv("clientes_reconstructed_full.csv", index=False)
files.download("clientes_reconstructed_full.csv")


Dataset shape: (1143, 13)

Reconstruction summary:


,Before,After,Recovered
customer_id,0,0,0
country_or_region,0,0,0
commercial_region,114,43,71
island,578,578,0
municipality,159,98,61
city,546,546,0
postal_area,71,71,0
postal_code,170,167,3
street,598,598,0
address,6,6,0


Total recovered: 135

--- Duplicate Check ---
Duplicated rows: 0
Duplicated customer IDs: 0
No duplicated customer IDs found.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

This Python cell creates a cleaner version of the dataset, keeping `customer_id` for joining with the transaction data and `postal_area`, `municipality`, `country_or_region`, and `commercial_region`, since I think these already provide enough geographical information.

After reconstructing the missing values in the previous step, I removed `street` (52.3% missing originally), `island` (50.6%), `city` (47.8%), and `postal_code` (14.9%) to reduce unnecessary categories, dimensionality, and the risk of overfitting. I used the original missing percentages because some values had already been recovered.

I initially considered keeping `address`, thinking that customers sharing the same address might have similar spending patterns. However, after checking the dataset, I found that only 33 customers shared an address out of more than 1,100. Since this gives us limited information to identify reliable patterns, I decided to remove `address` as well, especially considering its high cardinality.

The final dataset contains 5 columns, including `customer_id`, and keeps all 1,143 customers. Any remaining missing values will be handled in the next preprocessing step.

In [19]:

import pandas as pd
from google.colab import files

# 1. Read reconstructed dataset (NOT the original)
df = pd.read_csv("clientes_reconstructed_full.csv")

print("Reconstructed dataset shape:", df.shape)

# 2. Select columns to keep
columns_to_keep = [
    "customer_id",
    "postal_area",
    "municipality",
    "country_or_region",
    "commercial_region"
]

# 3. Remove unnecessary columns
df_cleaned = df[columns_to_keep].copy()

# 4. Check results
print("\nBefore column selection:", df.shape)
print("After column selection:", df_cleaned.shape)

print("\nRemaining missing values:")
print(df_cleaned.isna().sum())

display(df_cleaned.head())

# 5. Save and download final dataset
df_cleaned.to_csv("clientes_first_clean.csv", index=False)
files.download("clientes_first_clean.csv")


Reconstructed dataset shape: (1143, 13)

Before column selection: (1143, 13)
After column selection: (1143, 5)

Remaining missing values:
customer_id           0
postal_area          71
municipality         98
country_or_region     0
commercial_region    43
dtype: int64


,customer_id,postal_area,municipality,country_or_region,commercial_region
0,842570,NaN,NaN,Country CT11: Land of The Final Version,Region RG37: We Will See Later
1,770941,NaN,NaN,Country CT9: Land of The Final Version,NaN
2,827053,Postal area PA271: Hidden Sheet Sector,Municipality MU14: Council of Last Minute Edits,Country CT10: Land of The Final Version,Region RG17: We Will See Later
3,494648,Postal area PA271: Hidden Sheet Sector,Municipality MU14: Council of Last Minute Edits,Country CT10: Land of The Final Version,Region RG17: We Will See Later
4,429291,Postal area PA271: Hidden Sheet Sector,Municipality MU14: Council of Last Minute Edits,Country CT10: Land of The Final Version,Region RG17: We Will See Later


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

This Python cell replaces the remaining missing values in the geographical columns with `"Unknown"`, since they couldn't be reliably reconstructed in the previous steps. I decided to treat them as a separate category rather than deleting customers or assigning potentially incorrect geographical information.

In the reconstruction step, I recovered **135 missing values**, but only **132 were kept** in the final dataset (61 in `municipality` and 71 in `commercial_region`). The other 3 belonged to `postal_code`, which I removed during column selection.

Finally, I replaced the remaining **212 missing values** with `"Unknown"`, checked that no NaNs remained, and saved the dataset as `clientes_final.csv`, ready for the next steps of the project.

In [20]:
# 1. Read cleaned dataset
df = pd.read_csv("clientes_first_clean.csv")

# 2. Replace remaining missing values
geo_columns = [
    "postal_area",
    "municipality",
    "country_or_region",
    "commercial_region"
]

df[geo_columns] = df[geo_columns].fillna("Unknown")

# 3. Check results
print("Dataset shape:", df.shape)
print("\nRemaining missing values:")
print(df.isna().sum())

# 4. Save and download
df.to_csv("clientes_final.csv", index=False)
files.download("clientes_final.csv")


Dataset shape: (1143, 5)

Remaining missing values:
customer_id          0
postal_area          0
municipality         0
country_or_region    0
commercial_region    0
dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>